# Axion angle θ of one structure: inversion-parity indicator

For an inversion-symmetric insulator whose Chern numbers vanish, θ is fixed by the parities of the occupied bands at
the eight TRIMs (Z₄ indicator):

$$\eta_4 = \sum_{K} \frac{n_+^K - n_-^K}{2} \bmod 4,\qquad \theta = \pi\,\eta_4/2 \pmod{2\pi}\ \ (\eta_4 \in \{0, 2\}).$$

Odd η₄ means a nonzero Chern number (θ undefined). The model is loaded as in production (H from the `.chk`, no
hopping cut; `PIN_TO_ATOMS` for MnBi₂Te₄, bug #21), and the inversion matrix is built from the `.nnkp` projections:
each Wannier function maps to the one of the same (l, m_r, spin) at the inverted site, with sign (−1)^l. Valid for
a projection-only Wannierization (`num_iter = 0`), whose functions keep their trial orbitals' symmetry; the cell
checks D H(k) Dᵀ = H(−k) before using it.

Replaces the SCDM Chern–Simons 3-form that was here: its trial functions are almost inversion- and S-symmetric, so
for θ = π the gauge is singular at Γ (MnBi₂Te₄: σ_min 4e-5). The 3-form in a smooth gauge (designed or optimized
trials, exact connection) is in `archive/diagnostics/2026-10-14_cs_theta/`; on MnBi₂Te₄ it needs nk ≳ 16.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "axion_phonon/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
from axion_phonon import wannier
from axion_phonon.io import wannier90

In [2]:
PREFIX = "MnBi2Te4"
RUN = DATA / "MnBi2Te4/archive/base/lda/w90_sp_d_k4_210651"   # a projection-only Wannier run
N_OCC = 82                    # occupied Wannier bands (MnBi2Te4 116-WF set: 82; Y2Ir2O7 trial_02: 156)
PIN_TO_ATOMS = True           # True for MnBi2Te4 (bug #21); False for Y2Ir2O7
INVERSION_CENTRE = np.zeros(3)   # crystal coordinates of the inversion centre (MnBi2Te4: Mn1 at the origin)
NK_GAP = 6                    # mesh for the gap check

In [3]:
model = wannier.run_model(wannier90.read_run(RUN, PREFIX, overlaps=False), pin=PIN_TO_ATOMS)   # H(R), no cut
tau = np.asarray(model.orb_vecs, float)
H = lambda k: model.hamiltonian(np.atleast_2d(k), flatten_spin_axis=True)

# Wannier projections (site, l, m_r, spin) from the .nnkp, in Wannier-function order
lines = (RUN / f"{PREFIX}.nnkp").read_text().splitlines()
start = next(i for i, line in enumerate(lines) if line.strip() == "begin spinor_projections")
projections = []
for j in range(int(lines[start + 1])):
    a = lines[start + 2 + 3 * j].split()
    projections.append((np.array([float(x) for x in a[:3]]), int(a[3]), int(a[4]), int(lines[start + 4 + 3 * j].split()[0])))

# inversion r -> 2c - r on the Wannier basis
D = np.zeros((len(projections),) * 2)
for i, (site, l, mr, spin) in enumerate(projections):
    image = 2 * INVERSION_CENTRE - site
    j = [j for j, (s2, l2, mr2, sp2) in enumerate(projections)
         if (l2, mr2, sp2) == (l, mr, spin) and np.allclose((s2 - image + 0.5) % 1 - 0.5, 0, atol=1e-4)]
    assert len(j) == 1, f"Wannier function {i}: no unique inversion partner"
    D[j[0], i] = (-1) ** l
ks = np.random.default_rng(0).random((5, 3))
print(f"inversion: max |D H(k) D^T - H(-k)| = {max(np.abs(D @ H(k)[0] @ D.T - H(-k)[0]).max() for k in ks):.1e} eV")

grid = np.stack(np.meshgrid(*[np.arange(NK_GAP) / NK_GAP] * 3, indexing="ij"), -1).reshape(-1, 3)
E = np.linalg.eigvalsh(H(grid))
print(f"gap at N_OCC on {NK_GAP}^3: direct {np.min(E[:, N_OCC] - E[:, N_OCC - 1]) * 1e3:.1f} meV, "
      f"indirect {(E[:, N_OCC].min() - E[:, N_OCC - 1].max()) * 1e3:.1f} meV")

inversion: max |D H(k) D^T - H(-k)| = 4.5e-07 eV


gap at N_OCC on 6^3: direct 198.1 meV, indirect 198.1 meV


In [4]:
# parities at the TRIMs; in convention I, H(k - G) = Phi H(k) Phi^+ with Phi = diag(exp(2 pi i G.tau)), so the
# operator commuting with H(K) at a TRIM K (G = 2K) is Phi^+ D
eta = 0
for K in np.array(np.meshgrid([0, .5], [0, .5], [0, .5], indexing="ij")).reshape(3, -1).T:
    P = np.diag(np.exp(-2j * np.pi * tau @ (2 * K))) @ D
    e, U = np.linalg.eigh(H(K)[0])
    parity = np.linalg.eigvals(U[:, :N_OCC].conj().T @ P @ U[:, :N_OCC])
    n_minus = int(np.sum(parity.real < 0))
    eta += N_OCC - 2 * n_minus
    print(f"K = {K}: n+ = {N_OCC - n_minus:3d}, n- = {n_minus:3d}   "
          f"(|[P, H]| = {np.abs(P @ H(K)[0] - H(K)[0] @ P).max():.0e}, max |parity| - 1 = {np.abs(np.abs(parity) - 1).max():.0e})")
eta4 = (eta // 2) % 4
print(f"eta4 = {eta4}  ->  " + {0: "theta = 0", 2: "theta = pi"}.get(eta4, "odd: a nonzero Chern number, theta undefined"))

K = [0. 0. 0.]: n+ =  48, n- =  34   (|[P, H]| = 5e-07, max |parity| - 1 = 3e-14)


K = [0.  0.  0.5]: n+ =  41, n- =  41   (|[P, H]| = 4e-07, max |parity| - 1 = 7e-15)


K = [0.  0.5 0. ]: n+ =  41, n- =  41   (|[P, H]| = 5e-07, max |parity| - 1 = 7e-15)


K = [0.  0.5 0.5]: n+ =  46, n- =  36   (|[P, H]| = 4e-07, max |parity| - 1 = 8e-15)


K = [0.5 0.  0. ]: n+ =  41, n- =  41   (|[P, H]| = 4e-07, max |parity| - 1 = 7e-15)


K = [0.5 0.  0.5]: n+ =  46, n- =  36   (|[P, H]| = 5e-07, max |parity| - 1 = 1e-14)


K = [0.5 0.5 0. ]: n+ =  46, n- =  36   (|[P, H]| = 4e-07, max |parity| - 1 = 5e-15)


K = [0.5 0.5 0.5]: n+ =  41, n- =  41   (|[P, H]| = 5e-07, max |parity| - 1 = 8e-15)
eta4 = 2  ->  theta = pi
